# 1 - PDF 파일 준비

In [ ]:
'''
https://www.bok.or.kr/portal/bbs/P0002359/view.do?nttId=10096683&searchCnd=1&searchKwd=&depth2=200699&depth3=200066&depth=200066&pageUnit=10&pageIndex=1&programType=newsData&menuNo=200066&oldMenuNo=200066
'''

# 2. 환경 준비 (pass)

In [38]:
try:
    import pypdf
    print(f"pypdf: {pypdf.__version__}")
except ImportError:
    print("pypdf 미설치 → pip install pypdf")

pypdf: 6.9.2


# 3. Pinecone 인덱스 신규 생성

In [39]:
from dotenv import load_dotenv
import os
load_dotenv()

from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])

INDEX_NAME = "finance-bok"
NAMESPACE  = "bok-ns1"

# 기존 인덱스 없으면 생성
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"{INDEX_NAME} 생성 완료")
else:
    print(f"{INDEX_NAME} 이미 존재함")

# 상태 확인
bok_index = pc.Index(INDEX_NAME)
print(bok_index .describe_index_stats())

finance-bok 생성 완료
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '151',
                                    'content-type': 'application/json',
                                    'date': 'Tue, 14 Apr 2026 02:29:18 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '69',
                                    'x-pinecone-request-latency-ms': '68',
                                    'x-pinecone-response-duration-ms': '70'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


# 4. PDF 로딩 및 청킹
## 4.1. PDF 로딩

In [40]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "./data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf"

docs = PyPDFLoader(PDF_PATH).load()
# docs = loader.load()

print(f"총 페이지 수: {len(docs)}")
print(f"\n첫 페이지 내용 (앞 300자):\n{docs[0].page_content[:300]}")
print(f"\n메타데이터: {docs[0].metadata}")

# # 출력 예시
# {
#     'source': 'data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf',
#     'page': 0
# }

총 페이지 수: 93

첫 페이지 내용 (앞 300자):
경제전망 
 Indigo Book 
2026년 2월 
 
 
 
 
성장 2%대 반등, 부문별 온도차

메타데이터: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-02-26T16:48:20+09:00', 'author': 'A11', 'moddate': '2026-03-31T15:12:50+09:00', 'source': './data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf', 'total_pages': 93, 'page': 0, 'page_label': '1'}


In [41]:
# 추가 메타가 필요하면 직접 추가 가능
for doc in docs:
    doc.metadata['title'] = '2026년 2월 경제전망보고서'
    doc.metadata['year']  = 2026
    doc.metadata['month'] = 2
    del doc.metadata['producer']
    del doc.metadata['creator']
    del doc.metadata['author']
    del doc.metadata['creationdate']
    doc.metadata['page'] += 1
    del doc.metadata['page_label']
    

In [42]:
docs

[Document(metadata={'moddate': '2026-03-31T15:12:50+09:00', 'source': './data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf', 'total_pages': 93, 'page': 1, 'title': '2026년 2월 경제전망보고서', 'year': 2026, 'month': 2}, page_content='경제전망 \n Indigo Book \n2026년 2월 \n \n \n \n \n성장 2%대 반등, 부문별 온도차'),
 Document(metadata={'moddate': '2026-03-31T15:12:50+09:00', 'source': './data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf', 'total_pages': 93, 'page': 2, 'title': '2026년 2월 경제전망보고서', 'year': 2026, 'month': 2}, page_content=''),
 Document(metadata={'moddate': '2026-03-31T15:12:50+09:00', 'source': './data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf', 'total_pages': 93, 'page': 3, 'title': '2026년 2월 경제전망보고서', 'year': 2026, 'month': 2}, page_content='부문별 담당자 \n \n부 문 담 당 팀 담 당 자 \n<작성 총괄> 조사총괄팀 박창현 임웅지 \n장수정 강보민 한진수 \n국내외 여건 및 전망 \n1. 주요 여건 점검 \n\uf084 대외여건 \n글로벌 경제 국제종합팀 김보희 왕재곤 \n글로벌 반도체 경기 경기동향팀 이현아 김지현 \n미국 \n미국유럽경제팀 \n선진산 이진호 \n유로지역 강은서 \n중국 중국경제팀 이준호 전민근 류호정 \n일본, 아시아 신흥국 아태경제팀 전은총 김선중 박민재 \n국제유가 국제종합팀 김윤경 \n\uf084

## 4.2. 텍스트 분할

In [43]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

splits = splitter.split_documents(docs) #원본 문서의 메타 정보를 가져와 설정

print(f"총 청크 수: {len(splits)}")
print(f"\n첫 번째 청크:\n{splits[0].page_content}")
print(f"\n메타데이터: {splits[0].metadata}")
# # 출력 예시
# {
#     'source': 'data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf',
#     'page': 0   # 원본 페이지 번호 유지
# }

총 청크 수: 249

첫 번째 청크:
경제전망 
 Indigo Book 
2026년 2월 
 
 
 
 
성장 2%대 반등, 부문별 온도차

메타데이터: {'moddate': '2026-03-31T15:12:50+09:00', 'source': './data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf', 'total_pages': 93, 'page': 1, 'title': '2026년 2월 경제전망보고서', 'year': 2026, 'month': 2}


# 5. Pinecone 업서트
## 5.1. 임베딩 + 업서트

In [44]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = "finance-bok"
NAMESPACE  = "bok-ns1"

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

print("업서트 시작...")
vectorstore = PineconeVectorStore.from_documents(
    documents=splits,
    embedding=embedding_model ,
    index_name=INDEX_NAME,
    namespace=NAMESPACE
)
print(f"업서트 완료 — 총 {len(splits)}개 청크")

업서트 시작...
업서트 완료 — 총 249개 청크


## 5.2. 업서트 확인

In [45]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])
index = pc.Index(INDEX_NAME)
stats = index.describe_index_stats()
print(stats)

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '182',
                                    'content-type': 'application/json',
                                    'date': 'Tue, 14 Apr 2026 02:29:57 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '42',
                                    'x-pinecone-request-latency-ms': '39',
                                    'x-pinecone-response-duration-ms': '43'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'bok-ns1': {'vector_count': 249}},
 'storageFullness': 0.0,
 'total_vector_count': 249,
 'vector_type': 'dense'}


# 6. RAG 파이프라인 구성 및 테스트
## 6.1. retriever 생성 및 검색 테스트

In [46]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=embeddings,
    namespace=NAMESPACE
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3, "namespace": NAMESPACE}
)

# 검색 테스트
queries = [
    "2026년 경제성장률 전망은?",
    "물가 상승률 전망은?",
    "고용 시장 전망은?"
]

for query in queries:
    print(f"[ 질문 ] {query}")
    results = retriever.invoke(query)
    for i, r in enumerate(results):
        print(f"  {i+1}. (p.{r.metadata.get('page', '')}) {r.page_content[:100]}")
    print('-'*100)

[ 질문 ] 2026년 경제성장률 전망은?
  1. (p.44) 30 
 
<경제성장 전망1)> 
(전년동기대비, %) 
  2024 2025 2026e) 2027e) 
연간 상반 하반 연간 상반 하반 연간 연간 
GDP 성장률 2.0 0.3 
  2. (p.36) 22 
 
2. 거시경제 전망 
 
 
 
 
경제성장 
 
2-1. 금년중 국내경제는 美관세 영향, 건설투자의 더딘 회복에도 불구하고 반도체 경기 
개선세 확대, 예상보다 양호한
  3. (p.7) < 요약 1/8 > 
 
  
경제전망 요약 
 올해 우리 경제는 美관세 영향과 건설투자의 더딘 회복에도 불구하고 반도체 경기 
개선세 확대, 예상보다 양호한 세계경제 흐름 등에
----------------------------------------------------------------------------------------------------
[ 질문 ] 물가 상승률 전망은?
  1. (p.54) 1.9% 대비 0.1%p 높아졌다. 소비자물가 상승률의 경우에도 11월 전망1.9% 대비 0.1%p 높
아진 2.0%로 나타났다. 
 
시장의 국내 성장 및 물가 전망 모두 올해 
  2. (p.54) 40 
 
3. 전망의 리스크 평가 
    
주요 리스크 요인 
 
3-1. 향후 성장 전망경로에는 반도체 경기, 글로벌 통상환경, 국제금융시장 등과 관련
한 불확실성이 크며, 
  3. (p.12) < 요약 6/8 > 
6  
  
전망의 리스크 
 
 향후 성장 전망경로에는 반도체 경기, 글로벌 통상환경, 국제금융시장 등과 관련한 불확
실성이 크며, 물가의 경우 유가, 환
----------------------------------------------------------------------------------------------------
[ 질문 ] 고용 시장 전망은?
  1. (p.11) ▪ 상품수지는 반도체가격의 큰 폭 상승 등으로 흑자규모가 크게 늘어날 전망이다. 
서비

## 6.2. RAG 체인 구성

In [32]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_openai import ChatOpenAI

llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_template("""
    당신은 한국은행 경제전망 보고서를 기반으로 답변하는 금융 전문 어시스턴트입니다.
    아래 참고 문서를 바탕으로 질문에 정확하게 답하세요.
    답변 시 참조한 문서의 페이지 번호들을 답변 끝에 추가하고
    문서에 없는 내용은 "보고서에서 확인되지 않습니다"라고 답하세요.

    [참고문서]
    {context}

    [질문]
    {question}

    한글로 간결하고 정확하게 답변하세요.
""")

rag_chain = (
    RunnableParallel(
        context=retriever,
        question=RunnablePassthrough()
    )
    | prompt
    | llm
    | parser
)

# RAG 테스트
questions = [
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "소비자물가 상승률은 어떻게 전망하나요?",
    "수출 전망은 어떻게 되나요?"
]

for q in questions:
    print(f"[ Q ] {q}")
    answer = rag_chain.invoke(q)
    print(f"[ A ] {answer}")
    print()

[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
[ A ] 2026년 GDP 성장률 전망치는 2.0%입니다. (페이지 43)

[ Q ] 소비자물가 상승률은 어떻게 전망하나요?
[ A ] 소비자물가 상승률은 2026년 2.0%로 전망되고 있으며, 이는 11월 전망인 1.9% 대비 0.1%p 높아진 수치입니다. (페이지 11, 54)

[ Q ] 수출 전망은 어떻게 되나요?
[ A ] 수출은 견조한 흐름을 이어갈 것으로 전망되며, 2026년에는 통관 기준으로 7,952억 달러에 이를 것으로 예상됩니다. 이는 전년 대비 12.1% 증가한 수치입니다. (페이지 45)



# 7. RAG 파이프라인 동작 확인 및 답변 품질 평가
## 7.1. 검색 품질 확인

In [16]:
queries = [
    "2026년 경제성장률 전망은?",
    "소비자물가 상승률 전망은?",
    "수출 전망은 어떻게 되나요?"
]

for query in queries:
    print(f"[ 질문 ] {query}")
    results = retriever.invoke(query)
    print(f"검색된 청크 수: {len(results)}")
    for i, r in enumerate(results):
        print(f"  {i+1}. (p.{r.metadata.get('page', '')}) {r.page_content[:120]}")
    print()

[ 질문 ] 2026년 경제성장률 전망은?
검색된 청크 수: 3
  1. (p.43) 30 
 
<경제성장 전망1)> 
(전년동기대비, %) 
  2024 2025 2026e) 2027e) 
연간 상반 하반 연간 상반 하반 연간 연간 
GDP 성장률 2.0 0.3 1.6 1.0 2.4 1.6 2.0 
  2. (p.35) 22 
 
2. 거시경제 전망 
 
 
 
 
경제성장 
 
2-1. 금년중 국내경제는 美관세 영향, 건설투자의 더딘 회복에도 불구하고 반도체 경기 
개선세 확대, 예상보다 양호한 세계경제 흐름 등에 힘입어 성장률이
  3. (p.6) < 요약 1/8 > 
 
  
경제전망 요약 
 올해 우리 경제는 美관세 영향과 건설투자의 더딘 회복에도 불구하고 반도체 경기 
개선세 확대, 예상보다 양호한 세계경제 흐름 등에 힘입어 2.0% 성장할 전망이다. 

[ 질문 ] 소비자물가 상승률 전망은?
검색된 청크 수: 3
  1. (p.53) 1.9% 대비 0.1%p 높아졌다. 소비자물가 상승률의 경우에도 11월 전망1.9% 대비 0.1%p 높
아진 2.0%로 나타났다. 
 
시장의 국내 성장 및 물가 전망 모두 올해 2.0%중윗값로 소폭 상향 조정 
[
  2. (p.11) 대비 0.1%p 높아졌다. 소비자물가 상승률의 경우에도 11월 전망1.9% 대비 0.1%p 높
아진 2.0%로 나타났다. 
 
 
 
 
< 2026년 국내 성장률 및 소비자물가 상승률 전망 분포> 
GDP 성장률1
  3. (p.46) 33 
 
 
   
물가 
 
2-9. 소비자물가 상승률은 1월중 석유류가격 상승률이 큰 폭 낮아지고 농축수산
물가격 오름세도 둔화되면서 전월2.3%, 전년동월비보다 상당폭 낮은 2.0%로 나타났다. 석유
류가격은

[ 질문 ] 수출 전망은 어떻게 되나요?
검색된 청크 수: 3
  1. (p.35) 수출이 견조한 흐름을 이어가고 소비 등 내수가 회복되면서 성장세가 확대될 전망 
[그림 2.1] GDP 전망경로 [그림 2.2] 지출 부문별 기여도 [

## 7.2. 답변 품질 확인

In [17]:
# RAG 답변 vs 일반 LLM 답변 비교
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm_base = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

questions = [
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "소비자물가 상승률은 어떻게 전망하나요?",
    "수출 전망은 어떻게 되나요?"
]

for q in questions:
    print(f"[ Q ] {q}")

    # 일반 LLM (RAG 없음)
    base_answer = parser.invoke(llm_base.invoke(q))
    print(f"[ 일반 LLM ] {base_answer[:150]}")

    # RAG 기반 LLM
    rag_answer = rag_chain.invoke(q)
    print(f"[ RAG 답변 ] {rag_answer[:150]}")
    print()

[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
[ 일반 LLM ] 2026년의 GDP 성장률 전망치는 여러 경제 기관과 연구소에 따라 다를 수 있으며, 특정한 수치를 제공하기 위해서는 최신 경제 보고서나 예측 자료를 참조해야 합니다. 일반적으로 이러한 전망치는 경제 상황, 정책 변화, 글로벌 경제 동향 등에 따라 변동할 수 있습니다.
[ RAG 답변 ] 2026년 GDP 성장률 전망치는 2.0%입니다.

[ Q ] 소비자물가 상승률은 어떻게 전망하나요?
[ 일반 LLM ] 소비자물가 상승률 전망은 여러 요인에 따라 달라질 수 있습니다. 일반적으로 경제 성장률, 통화 정책, 원자재 가격, 공급망 문제, 그리고 글로벌 경제 상황 등이 주요한 영향을 미칩니다. 

2023년에는 많은 국가에서 인플레이션이 높은 수준을 유지하고 있으며, 이는 에
[ RAG 답변 ] 소비자물가 상승률은 2026년 2.0%로 전망되고 있습니다. 이는 11월 전망인 1.9% 대비 0.1%p 높아진 수치입니다.

[ Q ] 수출 전망은 어떻게 되나요?
[ 일반 LLM ] 수출 전망은 여러 요인에 따라 달라질 수 있습니다. 일반적으로 경제 성장률, 글로벌 수요, 환율 변동, 무역 정책, 그리고 특정 산업의 경쟁력 등이 중요한 요소로 작용합니다. 

2023년의 경우, 세계 경제의 회복세와 주요 국가들의 수요 증가가 긍정적인 영향을 미칠 
[ RAG 답변 ] 수출은 견조한 흐름을 이어갈 것으로 전망되며, 2026년에는 통관 기준으로 7,952억 달러에 이를 것으로 예상됩니다.



## 7.3. 종합 품질 점검

In [21]:
test_questions = [
    # 수치 확인용
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "2026년 소비자물가 상승률 전망치는?",
    "2026년 수출 전망 금액은 얼마인가요?",

    # 범위 밖 질문 (환각 테스트)
    "2030년 경제성장률 전망은?",
    "미국 연방준비제도의 금리 결정 일정은?",

    # 맥락 이해 확인
    "성장률이 2%대로 반등하는 주요 원인은?",
    "부문별 온도차가 발생하는 이유는?"
]

In [26]:
for q in test_questions:
    print(f"[ Q ] {q}")

    # 일반 LLM (RAG 없음)
    base_answer = parser.invoke(llm_base.invoke(q))
    print(f"[ 일반 LLM ] {base_answer[:150]}")

    # RAG 기반 LLM
    rag_answer = rag_chain.invoke(q)
    print(f"[ RAG 답변 ] {rag_answer[:150]}")
    print()

[ Q ] 2026년 유로 지역 경제 전망은?
[ 일반 LLM ] 2026년 유로 지역 경제 전망에 대한 구체적인 예측은 여러 요인에 따라 달라질 수 있습니다. 그러나 일반적으로 고려해야 할 몇 가지 주요 요소는 다음과 같습니다.

1. **금리 및 통화 정책**: 유럽 중앙은행(ECB)의 금리 결정과 통화 정책은 경제 성장에 큰 영
[ RAG 답변 ] 보고서에서 확인되지 않습니다.

[ Q ] 2022년부토 2024년까지 일본 경제 성장률 및 기여도는?
[ 일반 LLM ] 2022년부터 2024년까지 일본 경제 성장률 및 기여도에 대한 구체적인 수치는 여러 경제 지표와 예측에 따라 달라질 수 있습니다. 일본 경제는 여러 요인에 의해 영향을 받으며, 특히 COVID-19 팬데믹의 여파, 글로벌 공급망 문제, 인플레이션, 통화 정책, 그리고
[ RAG 답변 ] 보고서에서 확인되지 않습니다.

[ Q ] 2026년 유로경제 전망은??
[ 일반 LLM ] 2026년 유로존 경제 전망에 대한 구체적인 예측은 여러 요인에 따라 달라질 수 있습니다. 일반적으로 경제 전망은 다음과 같은 요소들에 의해 영향을 받습니다:

1. **금리 정책**: 유럽 중앙은행(ECB)의 금리 결정은 경제 성장과 인플레이션에 큰 영향을 미칩니다.
[ RAG 답변 ] 보고서에서 확인되지 않습니다.



In [30]:
custom_questions = [
    # 수치 확인용
    # "2026년 유로 지역 경제 전망은?",
    # "2022년부토 2024년까지 일본 경제 성장률 및 기여도는?",
    # "2026년 유로경제 전망은??",

    "AI가 반도체 경기에 미치는 영향은?"
    # "2030년 경제성장률 전망은?",
    # "미국 연방준비제도의 금리 결정 일정은?",

    # # 맥락 이해 확인
    # "성장률이 2%대로 반등하는 주요 원인은?",
    # "부문별 온도차가 발생하는 이유는?"
]

for q in custom_questions:
    print(f"[ Q ] {q}")
    answer = rag_chain.invoke(q)
    print(f"[ A ] {answer}")
    print()

[ Q ] AI가 반도체 경기에 미치는 영향은?
[ A ] AI의 진전은 반도체 수요를 구조적으로 확대하고 있으며, AI산업의 양적 및 질적 성장으로 인해 반도체 경기가 견조한 흐름을 지속할 것으로 예상됩니다. 특히, 빅테크 기업들의 선제적 투자와 공급자 우위 여건이 반도체 경기에 긍정적인 영향을 미치고 있습니다. 다만, 피지컬 AI의 빠른 확산과 AI 투자 조정 등 리스크 요소도 존재합니다.



## 7.4. 청킹 전략 비교 실습

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 전략 A — 작은 청크 (정밀 검색)
splitter_a = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

# 전략 B — 중간 청크 (권장)
splitter_b = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# 전략 C — 큰 청크 (문맥 유지)
splitter_c = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

for name, splitter in [("A(200)", splitter_a), ("B(500)", splitter_b), ("C(1000)", splitter_c)]:
    splits = splitter.split_documents(docs)
    print(f"전략 {name}: 청크 수 {len(splits)}개, 평균 길이 {sum(len(s.page_content) for s in splits)//len(splits)}자")